# Support Ticket → Department Classifier

**Goal:** given the text of a customer support ticket, predict which department should handle it.

**Dataset:** Multilingual Customer Support Tickets — Kaggle, `tobiasbueck/multilingual-customer-support-tickets`
~28,600 tickets, 16 columns. The label is `queue`, which holds the department name.

**Departments in the label:** Technical Support, Customer Service, Billing and Payments,
Product Support, IT Support, Returns and Exchanges, Sales and Pre-Sales, Human Resources.

> **This is topic classification, not sentiment analysis.** We are sorting tickets into
> departments, not measuring whether the customer is happy or angry. Worth stating up front
> because it changes which metrics matter.

---

### Pipeline map

| Steps | Phase | What happens |
|---|---|---|
| 1–4 | **Understand** | Setup, load, inspect, study the label |
| 5–10 | **Clean** | Build text field, handle missing values, kill duplicates, decide on language |
| 11–14 | **Prepare** | Rare classes, split, vectorize, handle imbalance |
| 15–16 | **Baseline** | Train 4 models, compare them fairly |
| 17–19 | **Improve** | Tune, analyse errors, fix confidence scores |
| 20 | **Finalize** | Test once, save, document limitations |

### One rule that runs through all of it

Anything learned from data — vocabulary, weights, scaling, resampling — is fitted on the
**training set only**. The moment test data influences training, the final score stops
meaning anything.

---

# Phase 1 — Understand the data
*Steps 1 to 4*

## Step 1 — Setup

Imports, plotting defaults, and one fixed random seed.

**Why the seed matters:** without it, every re-run shuffles the data differently and gives
slightly different numbers. You then can't tell whether a change actually improved the model
or whether you just got lucky. Fix it once, use it everywhere.

In [ ]:
# Step 1 — imports, seed, display settings

## Step 2 — Load the data

Download the dataset from Kaggle and put the CSV in a `data/` folder next to this notebook.

```
kaggle datasets download -d tobiasbueck/multilingual-customer-support-tickets
unzip multilingual-customer-support-tickets.zip -d data/
```

Load it into a dataframe and keep the raw version untouched, so we always have something
to compare back to after cleaning.

In [ ]:
# Step 2 — load CSV into df_raw

## Step 3 — Inspect what we actually have

Before touching anything, look at:

- **Shape** — how many rows (depth) and columns (width)
- **Column names and types** — what's text, what's categorical, what's a date
- **Missing values** — which columns have gaps, and how big
- **Unique counts** — a column with 1 unique value is useless; one with 28,000 is an ID

Surprises are much cheaper to find now than after training a model on them.

In [ ]:
# Step 3 — shape, dtypes, missing values, unique counts

## Step 4 — Study the label (class balance)

**This is the most important early check.**

Count how many tickets each department has. If one department has 8,000 tickets and another
has 200, a model can score high accuracy by *always guessing the big one* — while being
completely useless on the small ones.

Two things to write down here:

1. **The imbalance ratio** — biggest class divided by smallest
2. **The "always guess biggest" accuracy** — this is the floor. Any model that doesn't beat
   this number is worth nothing, no matter how impressive the accuracy looks.

Also plot it. A bar chart of tickets per department makes the problem obvious at a glance.

In [ ]:
# Step 4 — value_counts on the label, imbalance ratio, baseline floor

In [ ]:
# Step 4 (plot) — horizontal bar chart of tickets per department

---
# Phase 2 — Clean the data
*Steps 5 to 10*

## Step 5 — Build the text field

The model reads one block of text per ticket. We join `subject` and `body` together.

**Why both:** the subject often carries the strongest signal ("Invoice wrong" → Billing),
while the body carries the detail that resolves ambiguous cases.

Also record text length (characters and words) — we'll use it in the next step to find
tickets that are too short to be useful.

In [ ]:
# Step 5 — combine subject + body into a single text column, add length columns

## Step 6 — Drop rows we can't use

Two kinds of rows have to go:

- **No label** — we can't train on them or score them
- **Almost no text** — a ticket with two words carries no real signal and just adds noise

Print how many rows each rule removes. Nothing should disappear silently; if a rule removes
40% of the data, you want to know that immediately rather than discover it later.

In [ ]:
# Step 6 — drop rows with missing label or near-empty text, report counts

## Step 7 — Remove exact duplicates

**Why this matters more than it sounds:** if the same ticket text appears twice, and one copy
lands in training while the other lands in test, the model has effectively been handed the
answer key. The test score goes up and means nothing.

Remove exact repeats of the ticket text, keeping the first occurrence. Print a couple of
examples before dropping them — sometimes duplicates reveal a data collection bug worth knowing about.

In [ ]:
# Step 7 — find and drop exact duplicate texts

## Step 8 — Remove near-duplicates

Exact matching misses tickets that differ only by a customer name, an order number, or a
stray space but are otherwise identical. Those leak just as badly as exact copies.

**Approach:** normalise each text (lowercase, strip digits and punctuation, collapse
whitespace) to create a comparison key, then drop repeats of that key. Fast, and catches
most real-world near-duplication.

Report the total rows removed across Steps 6–8 as a percentage of the original — if it's
large, that's a finding worth mentioning in the write-up.

In [ ]:
# Step 8 — normalise text into a dedupe key, drop near-duplicate rows

## Step 9 — Decide what to do about language

This dataset mixes English, German, Spanish, French and Portuguese.

**Mixing languages by accident usually hurts,** because the model has to learn five
vocabularies at once from the same amount of data. Make it a deliberate choice:

| Option | Trade-off |
|---|---|
| **English only** | Fewer rows, cleaner signal, easier to interpret. Good first pass. |
| **All languages** | More data, harder problem, needs multilingual features to work well |

**Recommendation:** start English-only to get a trustworthy baseline, then re-run with all
languages later and compare. Set this as a switch at the top of the cell so it's easy to flip.

In [ ]:
# Step 9 — inspect language distribution, apply LANGUAGE_MODE filter

## Step 10 — Light text cleaning

**Common mistake: over-cleaning.** Aggressive stemming and stopword removal often *hurt*
text models because they throw away real signal. Resist the urge.

Only remove what is clearly noise *and* clearly leaky:

- **Email addresses** → replace with a placeholder
- **URLs** → replace with a placeholder
- **Long digit strings** (order IDs, ticket numbers) → replace with a placeholder

These can act as accidental giveaways tied to one department, which inflates the score
without the model learning anything real.

**Keep** casing and normal punctuation. TF-IDF handles those fine.

In [ ]:
# Step 10 — regex cleaning for emails, URLs, long numbers; show before/after

---
# Phase 3 — Prepare for modelling
*Steps 11 to 14*

## Step 11 — Handle very rare departments

If a department has only a handful of tickets, we can't train on it *or* score it honestly.
A stratified split needs at least a few examples of every class in every split.

**Two options:** drop those classes entirely, or merge them into an `Other` bucket.

**Prefer merging** — dropping silently pretends those tickets don't exist, which will
surprise someone in production when a real ticket arrives from that department.

Pick a minimum class size (30 is a reasonable starting point) and document the choice.

In [ ]:
# Step 11 — merge classes below MIN_CLASS_SIZE into 'Other', show final distribution

## Step 12 — Split into train / validation / test

Three separate pieces, each with a distinct job:

| Split | Share | Job |
|---|---|---|
| **Train** | 70% | The model learns from this |
| **Validation** | 15% | Compare models, tune settings |
| **Test** | 15% | Locked away — opened exactly once, at Step 20 |

**Stratified** means each department keeps the same share across all three splits, so a small
department doesn't accidentally end up entirely inside one of them.

**Critical ordering:** the split happens *before* any balancing or vectorizing. Doing it the
other way round leaks information from test into training and inflates the final number.

Verify afterwards that the class shares match across the three splits — they should be
nearly identical.

In [ ]:
# Step 12 — stratified train/val/test split, verify class shares match

## Step 13 — Turn text into numbers (TF-IDF)

Models need numbers, not words. TF-IDF scores each word by how often it appears in a ticket,
divided by how common it is across all tickets — so "invoice" counts for a lot and "the"
counts for almost nothing.

**Settings worth thinking about:**

- **`ngram_range=(1,2)`** — capture word pairs too, so "payment failed" is a feature, not
  just "payment" and "failed" separately
- **`min_df`** — ignore words appearing in very few tickets (usually typos)
- **`max_df`** — ignore words appearing in almost every ticket (no discriminating power)
- **`sublinear_tf`** — dampen the effect of a word repeated many times in one ticket

**The rule again:** `fit` on training data only. Validation and test get `transform` only.

In [ ]:
# Step 13 — fit TfidfVectorizer on train, transform val and test

## Step 14 — Deal with class imbalance

From Step 4 we know some departments are much bigger than others. Two ways to handle it:

**Option 1 — Class weights.** Tell the model that getting a small class wrong costs more.
No data is duplicated or discarded. Usually the better choice for text.

**Option 2 — Resampling.** Duplicate small-class rows (oversample) or delete big-class rows
(undersample). Risks overfitting on duplicated rows, or throwing away real information.

**Recommendation: class weights.** If you do try resampling later, apply it to the
**training set only** — never to validation or test, or the scores stop reflecting reality.

Print the computed weights so it's visible that small classes are getting boosted.

In [ ]:
# Step 14 — compute balanced class weights, display them against class counts

---
# Phase 4 — Baseline models
*Steps 15 to 16*

## Step 15 — Train 4 baseline models

**Always establish a floor before reaching for anything complicated.** If a simple model gets
you 90% of the way, an expensive model needs to justify itself.

| Model | Why it's in the lineup |
|---|---|
| **Dummy (most frequent)** | The "always guess the biggest department" score. The floor. Everything must beat it. |
| **Logistic Regression** | Standard strong baseline for text. Fast, interpretable, well-behaved probabilities. |
| **Linear SVM** | Often the best classical performer on sparse text data. |
| **Complement Naive Bayes** | Built specifically for imbalanced text. Extremely fast. |
| **Random Forest** | A non-linear comparison. Usually weaker on sparse text — worth confirming rather than assuming. |

**Score on validation, not test.** Test stays sealed until Step 20.

Record accuracy, macro F1, and weighted F1 for each.

In [ ]:
# Step 15 — train the 4 baselines + dummy, collect validation scores

## Step 16 — Compare the baselines properly

**Accuracy alone can lie here.** With imbalanced classes, a model can post good accuracy
while completely failing the small departments.

**Macro F1 is the primary metric.** It treats every department equally regardless of size.

**The diagnostic:** if accuracy is high but macro F1 is much lower, the model is quietly
ignoring the small classes. That gap is the imbalance problem showing up in the numbers.

Plot both metrics side by side per model, with the dummy floor drawn as a reference line.
Pick the winner by macro F1 and carry it into Step 17.

In [ ]:
# Step 16 — grouped bar chart of accuracy vs macro F1, mark the dummy floor, select best

---
# Phase 5 — Improve and diagnose
*Steps 17 to 19*

## Step 17 — Tune the best model

Search over the winner's settings to squeeze out more performance.

**Where to tune:** use **cross-validation on the training set** — not the validation set, and
definitely not the test set. Tuning repeatedly against validation slowly overfits to it, and
the validation score stops being a fair estimate.

**What to search:** the vectorizer settings and the model's regularisation strength together,
since they interact.

**`RandomizedSearchCV` over `GridSearchCV`** — it samples a fixed number of combinations
instead of trying every one. Far cheaper, and usually finds something just as good.

Compare the tuned score against the baseline score. If tuning gained you 0.3%, say so plainly
rather than presenting it as a breakthrough.

In [ ]:
# Step 17 — RandomizedSearchCV over pipeline, report best params and gain vs baseline

## Step 18 — Error analysis

A single score hides *where* the model fails. Three things to look at, in order:

**1. Per-class scores.** Which departments is it bad at? Small classes usually. Name the
worst three explicitly.

**2. Confusion matrix.** Which pairs does it mix up? Context matters here — Billing confused
with Payments is understandable and might be acceptable in production. Billing confused with
HR would signal something genuinely broken.

**3. Read real misclassified tickets.** Often the most useful five minutes in the whole
notebook. A large share of "errors" turn out to be genuinely ambiguous tickets that two
departments could legitimately own — which means the ceiling is lower than 100% and that's
fine.

Normalise the confusion matrix by row so each row sums to 1.0, otherwise the big classes
visually dominate and you can't see anything.

In [ ]:
# Step 18a — per-class classification report, identify weakest classes

In [ ]:
# Step 18b — normalised confusion matrix heatmap, list most-confused pairs

In [ ]:
# Step 18c — print a handful of actual misclassified tickets to read

## Step 19 — Check and fix the confidence scores (calibration)

Getting the answer right is one thing. Knowing *how sure* it is, is another.

**A well-calibrated model that says "85% confident" is right about 85% of the time when it
says that.** That matters in production — it's what lets you auto-route the confident tickets
and send the uncertain ones to a human.

**Measure first:** Expected Calibration Error (ECE) — how far stated confidence drifts from
actual accuracy. Lower is better. Also track log-loss.

**Then fix if needed:** isotonic regression or Platt scaling, fitted on validation.

**Then check the fix actually helped.** It doesn't always. Calibration corrects a distortion —
if the model was already honest, applying it can make things worse. Only keep the calibrated
version if ECE genuinely improved, and say which way it went.

Plot a reliability curve before and after, with dot size showing how many tickets fall in each
confidence bin — otherwise a wild-looking line from a 3-ticket bin misleads you.

In [ ]:
# Step 19a — compute ECE and log-loss before and after calibration, decide which to keep

In [ ]:
# Step 19b — reliability curve plot, before vs after

---
# Phase 6 — Finalize
*Step 20*

## Step 20 — Final evaluation, save, and document

**The test set gets opened once, here.** Whatever number comes out is the number we report.
No going back to tune after seeing it — that's the entire reason it was held back.

**Report:**
- Accuracy, macro F1, weighted F1, ECE
- How far it beat the "always guess biggest" floor from Step 4

**Plot:**
- Final confusion matrix on test
- The journey: floor → best baseline → tuned → final. Shows how much was actually gained.

**Save:**
- The fitted model to `artifacts/`
- A **model card** as JSON

**The model card is not optional.** It should state plainly:
- What the model does and what data it was trained on
- Which languages it covers (and therefore which it doesn't)
- Its weakest departments, by name and score
- Which department pairs it confuses, and whether that's acceptable
- That it was trained on one dataset from one source and needs revalidation on real traffic

Writing down what a model *can't* do is what separates a professional result from a demo.

In [ ]:
# Step 20a — predict on test set once, report all final metrics vs the floor

In [ ]:
# Step 20b — final confusion matrix + progress-from-floor chart

In [ ]:
# Step 20c — save model with joblib, write model_card.json with known limitations

---
## Done — and what comes next

**What we end up with:** a trained, tuned, calibration-checked department classifier,
evaluated once on data it never saw, with its weaknesses written down honestly.

**Sensible next experiments:**

1. **Re-run with `LANGUAGE_MODE = "all"`** and compare against the English-only numbers.
   Does more data beat cleaner data here?
2. **Try a transformer** (e.g. a multilingual sentence model) against this baseline. Only
   worth adopting if it *clearly* beats these numbers — otherwise the classical model wins
   on cost and speed.
3. **Set a confidence threshold.** Auto-route tickets above it, send the rest to a human.
   Step 19's calibration is what makes that threshold trustworthy.
4. **Compare against Jev** on this exact test set, if API access comes through. That would
   turn the article's vendor-published numbers into first-hand ones.